# AI Cybersecurity Unit 1 Mini Project

## AI-Based Social Media Threat Analysis and Classification

In this notebook, I complete all four tasks of my AI in Cyber Security lab assignment in sequence.

### Tasks
1. Workspace Bootstrap and Environment Setup
2. Cyber Security Threat Landscape Analysis
3. Basic Machine Learning-Based Threat Classification
4. Introduction to Deep Learning

I use the provided social-media dataset. The README describes `BinaryNumTarget` as the binary truth value of the statement, where `1 = True` and `0 = False`. I use the related tweet text for the classification part.

For Task 3, I use **Random Forest**.
For Task 4, I use **MLPClassifier**, which is a basic feed-forward Artificial Neural Network available in Scikit-learn.

I keep the comments and explanations simple so I can explain each step during my lab or viva.

## Task 1 — Workspace Bootstrap and Environment Setup

I use Google Colab for this project. I install or import the basic Python libraries needed for data analysis, visualization, and machine learning.

In [ ]:
# I install the main libraries required for this project.
!pip -q install numpy pandas scikit-learn matplotlib seaborn

In [ ]:
# I import the libraries that I will use throughout the notebook.
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("Environment setup completed successfully.")

Environment setup completed successfully.


### Load the dataset

I load the provided CSV file into a Pandas DataFrame. The expected file name is `Features_For_Traditional_ML_Techniques.csv`.

In [ ]:
# I load the provided dataset.
import pandas as pd

# locate the dataset automatically.

import pandas as pd

df = pd.read_csv("Features_For_Traditional_ML_Techniques.csv")

print(df.shape)
print(df["BinaryNumTarget"].value_counts(dropna=False))

FileNotFoundError: [Errno 2] No such file or directory: 'Features_For_Traditional_ML_Techniques.csv'

## Task 2 — Cyber Security Threat Landscape Analysis

In this task, I first understand the dataset and then perform basic EDA and data cleaning. I look at the structure of the data, the target distribution, missing values, duplicate rows, and some social-media activity features.

In [ ]:
# I display the first few records so I can understand the dataset.
display(df.head())

In [ ]:
# I check the data types and number of non-null values.
df.info()

In [ ]:
# I check the size of the dataset and the column names.
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

In [ ]:
# I check the basic statistical summary of numerical columns.
display(df.describe().T)

In [ ]:
# I check how many missing values are present in each column.
missing_values = df.isnull().sum().sort_values(ascending=False)

print("Total missing values:", int(missing_values.sum()))
display(missing_values[missing_values > 0].to_frame("Missing Values"))

In [ ]:
# I check for duplicate records.
duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)

In [ ]:
# I check the distribution of the target variable.
# 1 = True and 0 = False according to the dataset README.
print("BinaryNumTarget distribution:")
print(df["BinaryNumTarget"].value_counts(dropna=False))

target_percent = (
    df["BinaryNumTarget"]
    .value_counts(normalize=True, dropna=False)
    .mul(100)
    .round(2)
)

print("\nTarget distribution in percentage:")
print(target_percent)

In [ ]:
# I visualize the two target classes.
plt.figure(figsize=(7, 5))

sns.countplot(
    data=df,
    x="BinaryNumTarget"
)

plt.title("Target Class Distribution")
plt.xlabel("BinaryNumTarget")
plt.ylabel("Number of Records")
plt.xticks([0, 1], ["False", "True"])
plt.tight_layout()
plt.show()

### Activity feature analysis

I compare some social-media activity features between the two target classes. These plots help me understand whether user activity changes across the classes.

In [ ]:
# I compare retweet activity between the two classes.
if "retweets" in df.columns:
    plt.figure(figsize=(8, 5))
    sns.boxplot(
        data=df,
        x="BinaryNumTarget",
        y="retweets"
    )
    plt.title("Retweet Activity by Class")
    plt.xlabel("Class")
    plt.ylabel("Retweets")
    plt.xticks([0, 1], ["False", "True"])
    plt.tight_layout()
    plt.show()

In [ ]:
# I compare follower counts between the two classes.
if "followers_count" in df.columns:
    plt.figure(figsize=(8, 5))
    sns.boxplot(
        data=df,
        x="BinaryNumTarget",
        y="followers_count"
    )
    plt.title("Follower Count by Class")
    plt.xlabel("Class")
    plt.ylabel("Followers")
    plt.xticks([0, 1], ["False", "True"])
    plt.tight_layout()
    plt.show()

In [ ]:
# I compare a few activity features using their class-wise average.
activity_features = [
    "mentions",
    "quotes",
    "replies",
    "retweets",
    "favourites",
    "hashtags",
    "URLs"
]

available_activity = [
    col for col in activity_features
    if col in df.columns
]

if available_activity:
    activity_summary = (
        df.groupby("BinaryNumTarget")[available_activity]
        .mean()
        .rename(index={0: "False", 1: "True"})
    )

    display(activity_summary)

### Data cleaning

I clean the data before model training. I remove duplicate rows, handle missing target values, convert the target to integer form, and keep the original tweet and statement columns because I need them later for text classification.

I do not use target-related columns as model features because that could cause data leakage.

In [ ]:
# I remove exact duplicate rows.
before_duplicates = len(df)

df = df.drop_duplicates().reset_index(drop=True)

after_duplicates = len(df)

print("Rows before removing duplicates:", before_duplicates)
print("Rows after removing duplicates :", after_duplicates)
print("Duplicates removed              :", before_duplicates - after_duplicates)

In [ ]:
# I remove rows where the target value is missing.
df = df.dropna(subset=["BinaryNumTarget"]).copy()

# I make sure the target is stored as an integer.
df["BinaryNumTarget"] = df["BinaryNumTarget"].astype(int)

print("Target values after cleaning:")
print(df["BinaryNumTarget"].value_counts())

In [ ]:
# I fill missing tweet text with an empty string because the text model needs a valid string.
df["tweet"] = df["tweet"].fillna("").astype(str)

# I fill missing statement text with an empty string because it is used to create
# a statement-level train/test split.
df["statement"] = df["statement"].fillna("").astype(str)

print("Data cleaning completed.")
print("Final dataset shape:", df.shape)

### My observation from Task 2

From the EDA, I can see the class distribution and how social-media activity features differ between the two classes. I also checked missing values and duplicate records before moving to model training.

For the classification stage, I use the tweet text because it is directly related to the social-media content in the dataset.

## Task 3 — Basic Machine Learning-Based Threat Classification

I use **Random Forest** as my traditional machine-learning model.

Instead of randomly splitting individual tweets, I split by `statement`. This is important because several tweets can be related to the same statement. Keeping the same statement in only one side of the split reduces the chance of information leakage.

I convert tweet text into TF-IDF features and then reduce the feature size using TruncatedSVD before training Random Forest.

In [ ]:
from sklearn.model_selection import train_test_split

# I first create one label for each unique statement.
# This helps me keep tweets from the same statement in the same split.

statement_labels = (
    df.groupby("statement")["BinaryNumTarget"]
    .first()
    .reset_index()
)

print("Unique statements:", len(statement_labels))

# I split the statements into training and testing groups.

rf_train_statements, rf_test_statements = train_test_split(
    statement_labels,
    test_size=0.20,
    random_state=42,
    stratify=statement_labels["BinaryNumTarget"]
)

# I create sets of training and testing statements.

rf_train_statement_set = set(
    rf_train_statements["statement"]
)

rf_test_statement_set = set(
    rf_test_statements["statement"]
)

# I create the final training and testing data.

rf_train_df = df[
    df["statement"].isin(rf_train_statement_set)
].copy()

rf_test_df = df[
    df["statement"].isin(rf_test_statement_set)
].copy()

print("Training rows:", len(rf_train_df))
print("Testing rows :", len(rf_test_df))

print("\nTraining target:")
print(rf_train_df["BinaryNumTarget"].value_counts())

print("\nTesting target:")
print(rf_test_df["BinaryNumTarget"].value_counts())

# I check that the same statement is not present in both datasets.

overlap = (
    rf_train_statement_set &
    rf_test_statement_set
)

print("\nOverlapping statements:", len(overlap))

In [ ]:
# I separate tweet text and target values.

rf_train_text = (
    rf_train_df["tweet"]
    .fillna("")
    .astype(str)
)

rf_test_text = (
    rf_test_df["tweet"]
    .fillna("")
    .astype(str)
)

rf_y_train = (
    rf_train_df["BinaryNumTarget"]
    .astype(int)
)

rf_y_test = (
    rf_test_df["BinaryNumTarget"]
    .astype(int)
)

print("Training text:", len(rf_train_text))
print("Testing text :", len(rf_test_text))

print("\nTraining classes:")
print(rf_y_train.value_counts())

print("\nTesting classes:")
print(rf_y_test.value_counts())

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# I convert the tweet text into numerical TF-IDF features.

rf_tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=3,
    sublinear_tf=True,
    stop_words="english"
)

# I fit TF-IDF only on training text.
# The testing text is transformed using the already fitted vectorizer.

rf_train_tfidf = rf_tfidf.fit_transform(
    rf_train_text
)

rf_test_tfidf = rf_tfidf.transform(
    rf_test_text
)

print("TF-IDF training shape:", rf_train_tfidf.shape)
print("TF-IDF testing shape :", rf_test_tfidf.shape)

In [ ]:
y_train = rf_train_df["BinaryNumTarget"].astype(int)
y_test = rf_test_df["BinaryNumTarget"].astype(int)

# I separate the tweet text and target values for training and testing.
text_train = rf_train_df["tweet"].fillna("").astype(str)
text_test = rf_test_df["tweet"].fillna("").astype(str)

# I convert the tweet text into numerical TF-IDF features.
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=3,
    sublinear_tf=True,
    stop_words="english"
)

# I fit TF-IDF only on the training data to avoid test-data leakage.
X_train_tfidf = tfidf.fit_transform(text_train)
X_test_tfidf = tfidf.transform(text_test)

print("TF-IDF training shape:", X_train_tfidf.shape)
print("TF-IDF testing shape :", X_test_tfidf.shape)

In [ ]:
from sklearn.decomposition import TruncatedSVD

# I reduce the 5000 TF-IDF features to 50 components.
# This makes the data smaller and easier for the models to process.

rf_svd = TruncatedSVD(
    n_components=50,
    random_state=42
)

rf_train_features = rf_svd.fit_transform(
    rf_train_tfidf
)

rf_test_features = rf_svd.transform(
    rf_test_tfidf
)

print("SVD training shape:", rf_train_features.shape)
print("SVD testing shape :", rf_test_features.shape)

print(
    "Explained variance:",
    round(
        rf_svd.explained_variance_ratio_.sum(),
        4
    )
)

In [ ]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(
    n_components=50,
    random_state=42
)

X_train_svd = svd.fit_transform(X_train_tfidf)
X_test_svd = svd.transform(X_test_tfidf)

print("SVD training shape:", X_train_svd.shape)
print("SVD testing shape :", X_test_svd.shape)

print(
    "Explained variance:",
    round(svd.explained_variance_ratio_.sum(), 4)
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# I train my Random Forest classifier.

rf_model_final = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=2,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

rf_start = time.time()

rf_model_final.fit(
    rf_train_features,
    rf_y_train
)

rf_train_time = time.time() - rf_start

print(
    "Random Forest training completed in",
    round(rf_train_time, 2),
    "seconds."
)

print(
    "Classes learned by Random Forest:",
    rf_model_final.classes_
)

In [ ]:
# I use the trained Random Forest to predict the unseen test data.

rf_pred_final = rf_model_final.predict(
    rf_test_features
)

# I first check that the model learned both classes.

if len(rf_model_final.classes_) != 2:
    raise ValueError(
        f"Random Forest learned only these classes: "
        f"{rf_model_final.classes_}. "
        f"Both classes [0, 1] are required."
    )

# I get the probability of class 1 for ROC-AUC.

rf_prob_final = rf_model_final.predict_proba(
    rf_test_features
)[:, 1]

print("Random Forest prediction completed.")

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

rf_accuracy_final = accuracy_score(
    rf_y_test,
    rf_pred_final
)

rf_precision_final = precision_score(
    rf_y_test,
    rf_pred_final,
    zero_division=0
)

rf_recall_final = recall_score(
    rf_y_test,
    rf_pred_final,
    zero_division=0
)

rf_f1_final = f1_score(
    rf_y_test,
    rf_pred_final,
    zero_division=0
)

rf_auc_final = roc_auc_score(
    rf_y_test,
    rf_prob_final
)

rf_pr_auc_final = average_precision_score(
    rf_y_test,
    rf_prob_final
)

print("Random Forest Results")
print("---------------------")
print(f"Accuracy : {rf_accuracy_final:.4f}")
print(f"Precision: {rf_precision_final:.4f}")
print(f"Recall   : {rf_recall_final:.4f}")
print(f"F1-score : {rf_f1_final:.4f}")
print(f"ROC-AUC  : {rf_auc_final:.4f}")
print(f"PR-AUC   : {rf_pr_auc_final:.4f}")

print("\nClassification Report:")

print(
    classification_report(
        rf_y_test,
        rf_pred_final,
        target_names=["False", "True"],
        zero_division=0
    )
)

In [ ]:
# I visualize the Random Forest confusion matrix.

rf_cm = confusion_matrix(
    rf_y_test,
    rf_pred_final
)

plt.figure(figsize=(6, 5))

sns.heatmap(
    rf_cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["False", "True"],
    yticklabels=["False", "True"]
)

plt.title("Random Forest Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.tight_layout()
plt.show()

## Task 4 — Introduction to Deep Learning

For the deep-learning part, I use **MLPClassifier** from Scikit-learn.

MLP stands for Multi-Layer Perceptron. It is a basic feed-forward neural network, so it satisfies the lab requirement for a basic neural-network model.

I use the same TF-IDF/SVD representation and the same train/test data as Random Forest. Before training the MLP, I scale the features because neural networks generally work better when numerical inputs are on a similar scale.

In [ ]:
from sklearn.preprocessing import StandardScaler

# I use the same SVD features for my neural network.
# I scale them because neural networks usually work better
# when numerical inputs are on a similar scale.

mlp_scaler_final = StandardScaler()

mlp_train_features = mlp_scaler_final.fit_transform(
    rf_train_features
)

mlp_test_features = mlp_scaler_final.transform(
    rf_test_features
)

print(
    "MLP training shape:",
    mlp_train_features.shape
)

print(
    "MLP testing shape:",
    mlp_test_features.shape
)

In [ ]:
from sklearn.neural_network import MLPClassifier

# I use a simple MLP with two hidden layers.
# This is the basic neural-network model.

mlp_model_final = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    batch_size=256,
    learning_rate_init=0.001,
    max_iter=30,
    early_stopping=True,
    validation_fraction=0.20,
    n_iter_no_change=3,
    random_state=42
)

mlp_start = time.time()

mlp_model_final.fit(
    mlp_train_features,
    rf_y_train
)

mlp_train_time = time.time() - mlp_start

print(
    "MLPClassifier training completed in",
    round(mlp_train_time, 2),
    "seconds."
)

print(
    "Iterations used:",
    mlp_model_final.n_iter_
)

print(
    "Classes learned by MLP:",
    mlp_model_final.classes_
)

In [ ]:
# I generate predictions from my trained MLP.

mlp_pred_final = mlp_model_final.predict(
    mlp_test_features
)

# I check that the MLP learned both classes.

if len(mlp_model_final.classes_) != 2:
    raise ValueError(
        f"MLP learned only these classes: "
        f"{mlp_model_final.classes_}. "
        f"Both classes [0, 1] are required."
    )

mlp_prob_final = mlp_model_final.predict_proba(
    mlp_test_features
)[:, 1]

print("MLP prediction completed.")

In [ ]:
mlp_accuracy_final = accuracy_score(
    rf_y_test,
    mlp_pred_final
)

mlp_precision_final = precision_score(
    rf_y_test,
    mlp_pred_final,
    zero_division=0
)

mlp_recall_final = recall_score(
    rf_y_test,
    mlp_pred_final,
    zero_division=0
)

mlp_f1_final = f1_score(
    rf_y_test,
    mlp_pred_final,
    zero_division=0
)

mlp_auc_final = roc_auc_score(
    rf_y_test,
    mlp_prob_final
)

mlp_pr_auc_final = average_precision_score(
    rf_y_test,
    mlp_prob_final
)

print("MLPClassifier Results")
print("---------------------")
print(f"Accuracy : {mlp_accuracy_final:.4f}")
print(f"Precision: {mlp_precision_final:.4f}")
print(f"Recall   : {mlp_recall_final:.4f}")
print(f"F1-score : {mlp_f1_final:.4f}")
print(f"ROC-AUC  : {mlp_auc_final:.4f}")
print(f"PR-AUC   : {mlp_pr_auc_final:.4f}")

print("\nClassification Report:")

print(
    classification_report(
        rf_y_test,
        mlp_pred_final,
        target_names=["False", "True"],
        zero_division=0
    )
)

In [ ]:
# I plot the training loss to see how the MLP learned over the iterations.

plt.figure(figsize=(7, 5))

plt.plot(
    mlp_model_final.loss_curve_
)

plt.title("MLPClassifier Training Loss")
plt.xlabel("Iteration")
plt.ylabel("Loss")

plt.tight_layout()
plt.show()

In [ ]:
# I visualize the MLP confusion matrix.

mlp_cm_final = confusion_matrix(
    rf_y_test,
    mlp_pred_final
)

plt.figure(figsize=(6, 5))

sns.heatmap(
    mlp_cm_final,
    annot=True,
    fmt="d",
    cmap="Purples",
    xticklabels=["False", "True"],
    yticklabels=["False", "True"]
)

plt.title("MLPClassifier Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.tight_layout()
plt.show()

## Model Comparison

I now compare Random Forest and MLPClassifier using the same test data and the same reduced TF-IDF representation. This makes the comparison easier to understand because both models receive the same type of input.

In [ ]:
# I compare the two models using the same test dataset.

comparison_final = pd.DataFrame({
    "Model": [
        "Random Forest",
        "MLPClassifier"
    ],
    "Accuracy": [
        rf_accuracy_final,
        mlp_accuracy_final
    ],
    "Precision": [
        rf_precision_final,
        mlp_precision_final
    ],
    "Recall": [
        rf_recall_final,
        mlp_recall_final
    ],
    "F1-Score": [
        rf_f1_final,
        mlp_f1_final
    ],
    "ROC-AUC": [
        rf_auc_final,
        mlp_auc_final
    ],
    "PR-AUC": [
        rf_pr_auc_final,
        mlp_pr_auc_final
    ],
    "Training Time (sec)": [
        rf_train_time,
        mlp_train_time
    ]
})

display(
    comparison_final.round(4)
)

In [ ]:
from sklearn.metrics import roc_curve

# I compare the ROC curves of both models.

rf_fpr, rf_tpr, _ = roc_curve(
    rf_y_test,
    rf_prob_final
)

mlp_fpr, mlp_tpr, _ = roc_curve(
    rf_y_test,
    mlp_prob_final
)

plt.figure(figsize=(7, 5))

plt.plot(
    rf_fpr,
    rf_tpr,
    label=f"Random Forest (AUC = {rf_auc_final:.3f})"
)

plt.plot(
    mlp_fpr,
    mlp_tpr,
    label=f"MLPClassifier (AUC = {mlp_auc_final:.3f})"
)

plt.plot(
    [0, 1],
    [0, 1],
    "--",
    label="Random baseline"
)

plt.title("ROC Curve Comparison")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()

plt.tight_layout()
plt.show()

## My Conclusion

In this mini-project, I completed all four tasks in sequence.

For Task 1, I prepared the Colab environment and loaded the provided dataset.

For Task 2, I performed EDA and data cleaning. I checked the dataset structure, target distribution, missing values, duplicates, and social-media activity patterns.

For Task 3, I used Random Forest for the machine-learning classification task. I converted tweet text into TF-IDF features and reduced the feature size using TruncatedSVD.

For Task 4, I used MLPClassifier as my basic deep-learning model. I used the same processed features and the same statement-level test split so that the comparison is more consistent.

Finally, I compared both models using accuracy, precision, recall, F1-score, ROC-AUC, PR-AUC, confusion matrices, and training time.